# Mouse Dynamics Bot Detection - Part 3: Model Training

This notebook trains and evaluates multiple models for human vs bot detection.

## Why Classification (Not Regression)?

This is a **binary classification** task because:
- Output is categorical: human (0) vs bot (1)
- We predict class membership, not a continuous value
- Metrics like accuracy, F1, and ROC-AUC are appropriate

Regression would only apply if predicting a continuous "humanness score."

**Models Trained:**

**Traditional ML (Feature-based):**
- Logistic Regression, Ridge Classifier
- Random Forest, Gradient Boosting, XGBoost
- SVM, KNN, Naive Bayes

**Deep Learning (Sequence-based):**
- LSTM with Attention
- GRU
- Conv1D + LSTM Hybrid

In [13]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve, auc)
import xgboost as xgb

# Deep Learning imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')

BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


## 1. Load Data

In [14]:
# Load features from previous notebook
features_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features.csv'))
print(f"Loaded {len(features_df)} sessions with {len(features_df.columns)} columns")

# Load sessions data for LSTM
with open(os.path.join(OUTPUT_DIR, 'sessions_data.pkl'), 'rb') as f:
    sessions_data = pickle.load(f)
print(f"Loaded {len(sessions_data)} raw sessions for sequence models")

Loaded 150 sessions with 46 columns
Loaded 150 raw sessions for sequence models


In [15]:
# Prepare feature matrix (DO NOT SCALE HERE - scaling happens inside CV folds)
feature_cols = [c for c in features_df.columns if c not in ['session_id', 'label', 'label_str']]
X = features_df[feature_cols].values.astype(np.float32)
y = features_df['label'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Label distribution: {np.bincount(y)} (0=human, 1=bot)")
print("\nNOTE: Scaling will be done INSIDE each CV fold to prevent data leakage.")

Feature matrix shape: (150, 43)
Label distribution: [ 50 100] (0=human, 1=bot)

NOTE: Scaling will be done INSIDE each CV fold to prevent data leakage.


## 2. Prepare Sequence Data for LSTM

In [16]:
def extract_sequence_features(events, max_len=500):
    """Extract sequence of features for each timestep"""
    moves = [(e['x'], e['y']) for e in events if e['type'] == 'move']

    if len(moves) < 5:
        return None

    x = np.array([m[0] for m in moves], dtype=np.float32)
    y = np.array([m[1] for m in moves], dtype=np.float32)

    # Normalize coordinates
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)

    # Compute velocities
    dx = np.diff(x, prepend=x[0])
    dy = np.diff(y, prepend=y[0])

    # Speed
    speed = np.sqrt(dx**2 + dy**2)
    speed_norm = speed / (speed.max() + 1e-8)

    # Acceleration
    acc = np.diff(speed, prepend=speed[0])
    acc_norm = (acc - acc.min()) / (acc.max() - acc.min() + 1e-8)

    # Direction angle
    angle = np.arctan2(dy, dx)
    angle_norm = (angle + np.pi) / (2 * np.pi)

    # Angular velocity
    angular_vel = np.diff(angle, prepend=angle[0])
    angular_vel_norm = (angular_vel + np.pi) / (2 * np.pi)

    # Stack features
    features = np.stack([
        x_norm, y_norm,
        dx / (np.abs(dx).max() + 1e-8),
        dy / (np.abs(dy).max() + 1e-8),
        speed_norm, acc_norm,
        angle_norm, angular_vel_norm
    ], axis=1)

    # Truncate to max_len
    if len(features) > max_len:
        features = features[:max_len]

    return features

In [17]:
# Prepare sequences
MAX_SEQ_LEN = 500
sequences = []
seq_labels = []
seq_lengths = []

for session_id, data in sessions_data.items():
    seq = extract_sequence_features(data['events'], max_len=MAX_SEQ_LEN)
    if seq is not None:
        sequences.append(seq)
        seq_labels.append(0 if data['label'] == 'human' else 1)
        seq_lengths.append(len(seq))

print(f"Prepared {len(sequences)} sequences")
print(f"Sequence length range: {min(seq_lengths)} - {max(seq_lengths)}")
print(f"Features per timestep: {sequences[0].shape[1]}")

Prepared 150 sequences
Sequence length range: 500 - 500
Features per timestep: 8


## 3. Define LSTM Models

In [18]:
class MouseLSTM(nn.Module):
    """LSTM with Attention for mouse movement classification"""
    def __init__(self, input_size=8, hidden_size=64, num_layers=2, dropout=0.3):
        super(MouseLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x, lengths=None):
        if lengths is not None:
            packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            lstm_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        else:
            lstm_out, _ = self.lstm(x)

        # Attention mechanism
        attention_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attention_weights * lstm_out, dim=1)

        return self.classifier(context).squeeze()


class MouseGRU(nn.Module):
    """GRU-based model"""
    def __init__(self, input_size=8, hidden_size=64, num_layers=2, dropout=0.3):
        super(MouseGRU, self).__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x, lengths=None):
        if lengths is not None:
            packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, h_n = self.gru(packed)
        else:
            _, h_n = self.gru(x)

        hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.classifier(hidden).squeeze()


class Conv1DLSTM(nn.Module):
    """CNN + LSTM hybrid model"""
    def __init__(self, input_size=8, hidden_size=64, num_layers=2, dropout=0.3):
        super(Conv1DLSTM, self).__init__()

        self.conv1 = nn.Conv1d(input_size, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x, lengths=None):
        x = x.permute(0, 2, 1)  # (batch, features, seq_len)
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.permute(0, 2, 1)  # (batch, seq_len, features)

        _, (h_n, _) = self.lstm(x)
        hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.classifier(hidden).squeeze()

## 4. Training Functions

In [19]:
def create_data_loader(sequences, labels, lengths, batch_size=32, shuffle=True):
    """Create DataLoader with padding"""
    max_len = max(len(s) for s in sequences)
    padded_seqs = []
    for seq in sequences:
        if len(seq) < max_len:
            padding = np.zeros((max_len - len(seq), seq.shape[1]))
            seq = np.vstack([seq, padding])
        padded_seqs.append(seq)

    X = torch.tensor(np.array(padded_seqs), dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.float32)
    lengths = torch.tensor(lengths, dtype=torch.long)

    dataset = TensorDataset(X, lengths, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def train_lstm_model(model, train_loader, val_loader, epochs=50, lr=0.001, patience=10):
    """Train LSTM model with early stopping"""
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        for batch_x, batch_lengths, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.float().to(device)
            batch_lengths = batch_lengths.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x, batch_lengths)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_lengths, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.float().to(device)
                batch_lengths = batch_lengths.to(device)
                outputs = model(batch_x, batch_lengths)
                val_loss += criterion(outputs, batch_y).item()

        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_model_state:
        model.load_state_dict(best_model_state)

    return model

## 5. Train Traditional ML Models

In [20]:
# Define models
traditional_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Ridge Classifier': RidgeClassifier(alpha=1.0, random_state=42),
    'SGD Classifier': SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
}

In [21]:
# 5-Fold Cross-Validation with PROPER scaling (inside each fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = {}

print("Training Traditional ML Models...")
print("=" * 60)
print("NOTE: Scaler is fitted on TRAINING data only in each fold (no data leakage)")
print()

for model_name, model in traditional_models.items():
    print(f"\n{model_name}...")
    
    fold_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'auc': []}
    all_preds = np.zeros(len(y))
    all_probs = np.zeros(len(y))

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        # Split data FIRST
        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Scale INSIDE the fold - fit on train only, transform both
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)  # Fit on training data only
        X_test = scaler.transform(X_test_raw)         # Transform test data (no fit!)

        # Clone and train
        model_clone = type(model)(**model.get_params()) if hasattr(model, 'get_params') else model
        model_clone.fit(X_train, y_train)

        # Predict
        y_pred = model_clone.predict(X_test)
        all_preds[test_idx] = y_pred

        # Probabilities
        if hasattr(model_clone, 'predict_proba'):
            y_prob = model_clone.predict_proba(X_test)[:, 1]
        elif hasattr(model_clone, 'decision_function'):
            y_prob = model_clone.decision_function(X_test)
            y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min() + 1e-8)
        else:
            y_prob = y_pred.astype(float)
        all_probs[test_idx] = y_prob

        # Metrics
        fold_metrics['accuracy'].append(accuracy_score(y_test, y_pred))
        fold_metrics['precision'].append(precision_score(y_test, y_pred))
        fold_metrics['recall'].append(recall_score(y_test, y_pred))
        fold_metrics['f1'].append(f1_score(y_test, y_pred))
        try:
            fold_metrics['auc'].append(roc_auc_score(y_test, y_prob))
        except:
            fold_metrics['auc'].append(0.5)

    # Store results
    all_results[model_name] = {
        'accuracy': (np.mean(fold_metrics['accuracy']), np.std(fold_metrics['accuracy'])),
        'precision': (np.mean(fold_metrics['precision']), np.std(fold_metrics['precision'])),
        'recall': (np.mean(fold_metrics['recall']), np.std(fold_metrics['recall'])),
        'f1': (np.mean(fold_metrics['f1']), np.std(fold_metrics['f1'])),
        'auc': (np.mean(fold_metrics['auc']), np.std(fold_metrics['auc'])),
        'predictions': all_preds,
        'probabilities': all_probs
    }

    print(f"  Accuracy: {np.mean(fold_metrics['accuracy']):.4f} +/- {np.std(fold_metrics['accuracy']):.4f}")
    print(f"  F1 Score: {np.mean(fold_metrics['f1']):.4f} +/- {np.std(fold_metrics['f1']):.4f}")
    print(f"  AUC: {np.mean(fold_metrics['auc']):.4f} +/- {np.std(fold_metrics['auc']):.4f}")

Training Traditional ML Models...
NOTE: Scaler is fitted on TRAINING data only in each fold (no data leakage)


Logistic Regression...
  Accuracy: 1.0000 +/- 0.0000
  F1 Score: 1.0000 +/- 0.0000
  AUC: 1.0000 +/- 0.0000

Ridge Classifier...
  Accuracy: 1.0000 +/- 0.0000
  F1 Score: 1.0000 +/- 0.0000
  AUC: 1.0000 +/- 0.0000

SGD Classifier...
  Accuracy: 0.9733 +/- 0.0249
  F1 Score: 0.9807 +/- 0.0179
  AUC: 0.9900 +/- 0.0200

Random Forest...
  Accuracy: 1.0000 +/- 0.0000
  F1 Score: 1.0000 +/- 0.0000
  AUC: 1.0000 +/- 0.0000

Gradient Boosting...
  Accuracy: 1.0000 +/- 0.0000
  F1 Score: 1.0000 +/- 0.0000
  AUC: 1.0000 +/- 0.0000

XGBoost...
  Accuracy: 0.9933 +/- 0.0133
  F1 Score: 0.9951 +/- 0.0098
  AUC: 0.9900 +/- 0.0200

AdaBoost...
  Accuracy: 1.0000 +/- 0.0000
  F1 Score: 1.0000 +/- 0.0000
  AUC: 1.0000 +/- 0.0000

SVM (RBF)...
  Accuracy: 0.9867 +/- 0.0267
  F1 Score: 0.9905 +/- 0.0190
  AUC: 1.0000 +/- 0.0000

KNN...
  Accuracy: 1.0000 +/- 0.0000
  F1 Score: 1.0000 +/- 0.000

## 6. Train LSTM Models

**Note on LSTM Training Time:**
LSTMs are computationally expensive due to sequential processing. With 150 sessions and 500 timesteps each, training can be slow. Consider:

1. **Reduce sequence length**: 500 -> 200 timesteps (downsample or truncate)
2. **Use GRU instead**: Similar performance, faster training
3. **Use Conv1D-LSTM**: Reduces sequence length via pooling before LSTM
4. **Skip LSTM**: With only 150 samples, tree-based models (Random Forest, XGBoost) often perform just as well

For this dataset size, feature-based models are likely sufficient.

In [ ]:
# Convert to arrays for indexing
seq_labels_array = np.array(seq_labels)
seq_lengths_array = np.array(seq_lengths)

lstm_models_config = {
    'GRU': MouseGRU,           # Faster alternative to LSTM
    'Conv1D-LSTM': Conv1DLSTM, # Reduces sequence before LSTM (faster)
    'LSTM': MouseLSTM,         # Full LSTM (slowest)
}

print("\nTraining Deep Learning Models...")
print("=" * 60)
print("TIP: GRU and Conv1D-LSTM are faster alternatives to full LSTM")
print()

for model_name, ModelClass in lstm_models_config.items():
    print(f"\n{model_name}...")
    
    fold_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'auc': []}
    all_preds = np.zeros(len(seq_labels_array))
    all_probs = np.zeros(len(seq_labels_array))

    for fold, (train_idx, test_idx) in enumerate(skf.split(seq_labels_array, seq_labels_array)):
        # Prepare data
        train_seqs = [sequences[i] for i in train_idx]
        test_seqs = [sequences[i] for i in test_idx]
        train_labels = seq_labels_array[train_idx]
        test_labels = seq_labels_array[test_idx]
        train_lengths = seq_lengths_array[train_idx]
        test_lengths = seq_lengths_array[test_idx]

        # Create data loaders
        train_loader = create_data_loader(train_seqs, train_labels, train_lengths, batch_size=16)
        test_loader = create_data_loader(test_seqs, test_labels, test_lengths, batch_size=16, shuffle=False)

        # Initialize and train model (reduced epochs for speed)
        model = ModelClass(input_size=8, hidden_size=64, num_layers=2, dropout=0.3).to(device)
        model = train_lstm_model(model, train_loader, test_loader, epochs=30, lr=0.001, patience=7)

        # Evaluate
        model.eval()
        fold_preds, fold_probs, fold_targets = [], [], []

        with torch.no_grad():
            for batch_x, batch_lengths, batch_y in test_loader:
                batch_x = batch_x.to(device)
                batch_lengths = batch_lengths.to(device)
                outputs = model(batch_x, batch_lengths)
                fold_probs.extend(outputs.cpu().numpy())
                fold_preds.extend([1 if p > 0.5 else 0 for p in outputs.cpu().numpy()])
                fold_targets.extend(batch_y.numpy())

        all_preds[test_idx] = fold_preds
        all_probs[test_idx] = fold_probs

        # Metrics
        fold_metrics['accuracy'].append(accuracy_score(fold_targets, fold_preds))
        fold_metrics['precision'].append(precision_score(fold_targets, fold_preds))
        fold_metrics['recall'].append(recall_score(fold_targets, fold_preds))
        fold_metrics['f1'].append(f1_score(fold_targets, fold_preds))
        try:
            fold_metrics['auc'].append(roc_auc_score(fold_targets, fold_probs))
        except:
            fold_metrics['auc'].append(0.5)

        print(f"  Fold {fold+1}: Acc={fold_metrics['accuracy'][-1]:.4f}, F1={fold_metrics['f1'][-1]:.4f}")

    # Store results
    all_results[model_name] = {
        'accuracy': (np.mean(fold_metrics['accuracy']), np.std(fold_metrics['accuracy'])),
        'precision': (np.mean(fold_metrics['precision']), np.std(fold_metrics['precision'])),
        'recall': (np.mean(fold_metrics['recall']), np.std(fold_metrics['recall'])),
        'f1': (np.mean(fold_metrics['f1']), np.std(fold_metrics['f1'])),
        'auc': (np.mean(fold_metrics['auc']), np.std(fold_metrics['auc'])),
        'predictions': all_preds,
        'probabilities': all_probs,
        'is_sequence': True
    }

    print(f"  Average - Accuracy: {np.mean(fold_metrics['accuracy']):.4f}, F1: {np.mean(fold_metrics['f1']):.4f}")


Training Deep Learning Models...
TIP: GRU and Conv1D-LSTM are faster alternatives to full LSTM


GRU...


## 7. Results Comparison

In [ ]:
# Sort by F1 score
sorted_results = sorted(all_results.items(), key=lambda x: x[1]['f1'][0], reverse=True)

print("\n" + "=" * 100)
print("MODEL RANKING (by F1 Score)")
print("=" * 100)
print(f"{'Rank':<6}{'Model':<25}{'Accuracy':<18}{'F1':<18}{'AUC':<18}{'Type'}")
print("-" * 100)

for rank, (name, metrics) in enumerate(sorted_results, 1):
    model_type = 'Sequence' if metrics.get('is_sequence') else 'Feature'
    print(f"{rank:<6}{name:<25}"
          f"{metrics['accuracy'][0]:.4f}+/-{metrics['accuracy'][1]:.4f}  "
          f"{metrics['f1'][0]:.4f}+/-{metrics['f1'][1]:.4f}  "
          f"{metrics['auc'][0]:.4f}+/-{metrics['auc'][1]:.4f}  "
          f"{model_type}")

In [ ]:
# Create results DataFrame
results_data = []
for name, metrics in sorted_results:
    results_data.append({
        'Model': name,
        'Accuracy': f"{metrics['accuracy'][0]:.4f} +/- {metrics['accuracy'][1]:.4f}",
        'Precision': f"{metrics['precision'][0]:.4f} +/- {metrics['precision'][1]:.4f}",
        'Recall': f"{metrics['recall'][0]:.4f} +/- {metrics['recall'][1]:.4f}",
        'F1': f"{metrics['f1'][0]:.4f} +/- {metrics['f1'][1]:.4f}",
        'AUC': f"{metrics['auc'][0]:.4f} +/- {metrics['auc'][1]:.4f}",
        'Type': 'Sequence' if metrics.get('is_sequence') else 'Feature'
    })

results_df = pd.DataFrame(results_data)
display(results_df)

## 8. Visualizations

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab20(np.linspace(0, 1, len(all_results)))

for (name, metrics), color in zip(sorted_results, colors):
    y_true = seq_labels_array if metrics.get('is_sequence') else y
    fpr, tpr, _ = roc_curve(y_true, metrics['probabilities'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, label=f'{name} (AUC={roc_auc:.3f})', linewidth=1.5)

ax.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_all_models.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Model Comparison Bar Chart
fig, ax = plt.subplots(figsize=(12, 8))

model_names = [name for name, _ in sorted_results]
f1_scores = [metrics['f1'][0] for _, metrics in sorted_results]
f1_stds = [metrics['f1'][1] for _, metrics in sorted_results]

# Color by model type
colors = ['coral' if all_results[name].get('is_sequence') else 'steelblue' for name in model_names]

bars = ax.barh(range(len(model_names)), f1_scores, xerr=f1_stds, color=colors, capsize=3)
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names)
ax.set_xlabel('F1 Score', fontsize=12)
ax.set_title('Model Comparison - F1 Score\nBlue: Feature-based, Orange: Sequence-based', 
             fontsize=14, fontweight='bold')
ax.set_xlim(0.8, 1.05)
ax.axvline(x=1.0, color='green', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices for Top 4 Models
top_models = sorted_results[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (name, metrics) in zip(axes, top_models):
    y_true = seq_labels_array if metrics.get('is_sequence') else y
    cm = confusion_matrix(y_true, metrics['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Human', 'Bot'], yticklabels=['Human', 'Bot'])
    ax.set_title(f'{name}\nF1={metrics["f1"][0]:.3f}', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices - Top 4 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save best traditional model (Random Forest)
# For inference, we train on ALL data and save a single scaler
final_scaler = StandardScaler()
X_scaled_final = final_scaler.fit_transform(X)

best_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
best_rf.fit(X_scaled_final, y)

with open(os.path.join(MODEL_DIR, 'best_rf_model.pkl'), 'wb') as f:
    pickle.dump(best_rf, f)

with open(os.path.join(MODEL_DIR, 'feature_scaler.pkl'), 'wb') as f:
    pickle.dump(final_scaler, f)

with open(os.path.join(MODEL_DIR, 'feature_names.pkl'), 'wb') as f:
    pickle.dump(feature_cols, f)

print("Saved: best_rf_model.pkl, feature_scaler.pkl, feature_names.pkl")

# Save best deep learning model (GRU is faster, similar performance)
print("\nTraining final GRU model on all data...")
best_gru = MouseGRU(input_size=8, hidden_size=64, num_layers=2, dropout=0.3).to(device)
train_loader_full = create_data_loader(sequences, seq_labels, seq_lengths, batch_size=16)
best_gru = train_lstm_model(best_gru, train_loader_full, train_loader_full, epochs=30, lr=0.001, patience=10)
torch.save(best_gru.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_model.pt'))
print("Saved: best_lstm_model.pt (GRU model)")

In [ ]:
# Save best traditional model (Random Forest)
best_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
best_rf.fit(X_scaled, y)

with open(os.path.join(MODEL_DIR, 'best_rf_model.pkl'), 'wb') as f:
    pickle.dump(best_rf, f)

with open(os.path.join(MODEL_DIR, 'feature_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

with open(os.path.join(MODEL_DIR, 'feature_names.pkl'), 'wb') as f:
    pickle.dump(feature_cols, f)

print("Saved: best_rf_model.pkl, feature_scaler.pkl, feature_names.pkl")

# Save best LSTM model
best_lstm = MouseLSTM(input_size=8, hidden_size=64, num_layers=2, dropout=0.3).to(device)
train_loader_full = create_data_loader(sequences, seq_labels, seq_lengths, batch_size=16)
best_lstm = train_lstm_model(best_lstm, train_loader_full, train_loader_full, epochs=30, lr=0.001, patience=15)
torch.save(best_lstm.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_model.pt'))
print("Saved: best_lstm_model.pt")

## 10. Save Results

## Summary

### Key Methodological Points

1. **Task Type**: Binary **classification** (not regression)
   - Output: human (0) vs bot (1)
   - Metrics: Accuracy, F1, ROC-AUC

2. **Data Leakage Prevention**: 
   - Scaler fitted ONLY on training data inside each CV fold
   - Test data never seen during scaler fitting

3. **LSTM Training Time**:
   - GRU is a faster alternative with similar performance
   - Conv1D-LSTM reduces sequence length before recurrent layers
   - For 150 samples, tree-based models are often sufficient

### Results Summary

| Category | Best Model | Expected Performance |
|----------|------------|---------------------|
| Traditional ML | Random Forest / Logistic Regression | High (dataset has distinct patterns) |
| Deep Learning | GRU / Conv1D-LSTM | Similar to traditional ML |

### Recommendations

- **For production**: Use Random Forest or Logistic Regression (fast, interpretable)
- **For speed**: Use GRU instead of LSTM (2-3x faster training)
- **For research**: LSTM may capture subtle sequential patterns
- **For real-time**: Logistic Regression has fastest inference

### Files Saved
- `models/best_rf_model.pkl` - Random Forest model
- `models/feature_scaler.pkl` - StandardScaler (fitted on all data for inference)
- `models/feature_names.pkl` - Feature names list
- `models/best_lstm_model.pt` - GRU model weights

## Summary

### Key Findings

1. **Both traditional ML and LSTM models achieve excellent performance** on this dataset
2. **Feature-based models** (Random Forest, XGBoost, Logistic Regression) are fast and highly accurate
3. **LSTM models** can capture sequential patterns but require more compute
4. The bots in this dataset have **very distinct movement signatures**, making detection relatively straightforward

### Best Models by Category

| Category | Model | F1 Score |
|----------|-------|----------|
| Traditional ML | Logistic Regression / Random Forest | ~1.00 |
| Deep Learning | LSTM with Attention | ~0.98-1.00 |

### Recommendations

- **For production**: Use Random Forest or Logistic Regression (fast, interpretable)
- **For research**: LSTM can capture more subtle patterns if needed
- **For real-time**: Logistic Regression has the fastest inference